# 광고 문구 생성 검증

고정 샘플 제품으로 `generate_ad_strategy()` 실행 → format / language / relevance 자동 체크 → 결과 직접 확인

---

## 0. 환경 설정

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"  # Jupyter async 환경 허용
import django
django.setup()

## 1. 샘플 제품 설정

In [2]:
SAMPLE = {
    "product_name": "레티놀 5000샷 브라이트닝 부스터 세럼",
    "category":     "스킨케어",
    "ingredients":  "나이아신아마이드, 히알루론산, 레티놀, 스피큘, 콜라겐",
    "effects":      "미백, 주름개선, 피부결 개선, 장벽 강화, 수분 공급",
}

COUNTRIES = ["US", "JP"]

print("샘플 제품:", SAMPLE)

샘플 제품: {'product_name': '레티놀 5000샷 브라이트닝 부스터 세럼', 'category': '스킨케어', 'ingredients': '나이아신아마이드, 히알루론산, 레티놀, 스피큘, 콜라겐', 'effects': '미백, 주름개선, 피부결 개선, 장벽 강화, 수분 공급'}


---
## 2. 광고 전략 생성

In [3]:
from market_api.services.ad_strategy import generate_ad_strategy

results = {}
for country in COUNTRIES:
    print(f"[{country}] 생성 중...")
    results[country] = generate_ad_strategy(
        product_name=SAMPLE["product_name"],
        category=SAMPLE["category"],
        ingredients=SAMPLE["ingredients"],
        effects=SAMPLE["effects"],
        country=country,
    )
    print(f"  완료")

print("\n생성 완료")

[US] 생성 중...
  완료
[JP] 생성 중...
  완료

생성 완료


---
## 3. 자동 체크

In [4]:
# format / language 자동 체크
REQUIRED_FIELDS = ["brand_concept", "concept_reasoning", "key_messages", "ad_copies", "detailed_insight"]

def detect_language(text: str) -> str:
    """텍스트가 일본어/영어인지 간단 판별"""
    jp_chars = sum(1 for c in text if '\u3040' <= c <= '\u30ff' or '\u4e00' <= c <= '\u9fff')
    return "일본어" if jp_chars > 5 else "영어/기타"

EXPECTED_LANG = {"US": "영어/기타", "JP": "일본어"}

print(f"{'항목':<20} {'format':>8}  {'language':>10}  {'오류'}")
print("─" * 60)

for country, result in results.items():
    # format 체크
    if "error" in result:
        print(f"{country:<20} {'FAIL':>8}  {'─':>10}  {result['error']}")
        continue

    missing = [f for f in REQUIRED_FIELDS if f not in result]
    format_ok = "OK" if not missing else f"FAIL({','.join(missing)})"

    # language 체크 (brand_concept + 첫 번째 카피 헤드라인)
    copy_text = result.get("brand_concept", "")
    if result.get("ad_copies"):
        copy_text += " " + result["ad_copies"][0].get("headline", "")
    detected = detect_language(copy_text)
    expected = EXPECTED_LANG[country]
    lang_ok = "OK" if detected == expected else f"FAIL(expected {expected}, got {detected})"

    print(f"{country:<20} {format_ok:>8}  {lang_ok:>10}")

항목                     format    language  오류
────────────────────────────────────────────────────────────
US                         OK          OK
JP                         OK          OK


---
## 4. 결과 직접 확인

In [5]:
for country, result in results.items():
    if "error" in result:
        print(f"[{country}] 오류: {result['error']}")
        continue

    print(f"\n{'═' * 55}")
    print(f"  [{country}] 광고 전략 결과")
    print(f"{'═' * 55}")

    print(f"\n[브랜드 컨셉]")
    print(f"  {result.get('brand_concept', '')}")

    print(f"\n[컨셉 선정 이유]")
    print(f"  {result.get('concept_reasoning', '')}")

    print(f"\n[핵심 메시지]")
    for i, msg in enumerate(result.get('key_messages', []), 1):
        print(f"  {i}. {msg}")

    print(f"\n[광고 카피]")
    for i, copy in enumerate(result.get('ad_copies', []), 1):
        print(f"  [{i}] {copy.get('headline', '')}")
        print(f"      {copy.get('body_text', '')}")

    print(f"\n[마케팅 인사이트]")
    print(f"  {result.get('detailed_insight', '')}")

    stats = result.get('ad_stats', {})
    print(f"\n[광고 데이터 기반] 총 {stats.get('total_ads', 0)}건, 브랜드 {stats.get('brand_count', 0)}개")


═══════════════════════════════════════════════════════
  [US] 광고 전략 결과
═══════════════════════════════════════════════════════

[브랜드 컨셉]
  Elevate Your Glow with Advanced Brightening and Anti-Aging Care

[컨셉 선정 이유]
  미국 시장에서 레티놀과 나이아신아마이드는 여전히 인기 있는 성분이며, 소비자들은 간단하면서도 효과적인 스킨케어 루틴을 선호합니다. 레티놀 5000샷 브라이트닝 부스터 세럼은 이러한 트렌드에 부합하며, 미백과 주름 개선, 피부결 개선을 동시에 제공하는 멀티태스킹 제품으로 차별화할 수 있습니다.

[핵심 메시지]
  1. 고농축 레티놀과 나이아신아마이드로 피부를 밝고 매끄럽게.
  2. 히알루론산과 콜라겐으로 깊은 보습과 탄력 제공.
  3. 스피큘 성분으로 피부 장벽 강화 및 건강한 피부 유지.

[광고 카피]
  [1] Unleash Radiance with Retinol 5000 Shot
      Transform your skin with our powerful brightening booster serum. Packed with retinol and niacinamide, it targets dark spots and fine lines for a luminous complexion.
  [2] Achieve Youthful Skin with Multi-Tasking Serum
      Our Retinol 5000 Shot Brightening Booster Serum combines hydration and anti-aging benefits. Experience smoother, firmer skin with every drop.

[마케팅 인사이트]
  미국 스킨케어 시장에서는 간단하면서도 효과적인 제품이 주목받고 있으며, 특히 레티놀과 나이아신아마이드가 포함

---
## 5. relevance 체크 (AI 채점)

---
## 5. N회 반복 통계

In [6]:
import json
from collections import Counter
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from market_api.services.ad_strategy import generate_ad_strategy

N_RUNS = 10  # 반복 횟수
client = OpenAI()

REQUIRED_FIELDS = ["brand_concept", "concept_reasoning", "key_messages", "ad_copies", "detailed_insight"]

RELEVANCE_PROMPT = """아래 제품 정보와 생성된 광고 전략을 보고 relevance를 5.0 만점으로 채점하세요.
relevance: 제품의 성분·효능이 광고 카피와 핵심 메시지에 실제로 반영된 정도
반드시 JSON만 반환하세요: {"score": 4.2, "reason": "근거 1~2문장"}"""

def detect_language(text: str) -> str:
    jp_chars = sum(1 for c in text if '\u3040' <= c <= '\u30ff' or '\u4e00' <= c <= '\u9fff')
    return "일본어" if jp_chars > 5 else "영어/기타"

EXPECTED_LANG = {"US": "영어/기타", "JP": "일본어"}

def score_relevance(result: dict) -> float:
    user_msg = f"제품 정보: {json.dumps(SAMPLE, ensure_ascii=False)}\n\n광고 전략: {json.dumps(result, ensure_ascii=False)}"
    resp = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": RELEVANCE_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content.strip()).get("score", 0.0)

def run_once(country):
    result = generate_ad_strategy(
        product_name=SAMPLE["product_name"],
        category=SAMPLE["category"],
        ingredients=SAMPLE["ingredients"],
        effects=SAMPLE["effects"],
        country=country,
    )
    return result

# 국가별 통계 수집
all_stats = {}

for country in COUNTRIES:
    print(f"\n[{country}] {N_RUNS}회 실행 중...")
    run_results = []

    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(run_once, country) for _ in range(N_RUNS)]
        for i, future in enumerate(as_completed(futures)):
            run_results.append(future.result())
            print(f"  [{i+1}/{N_RUNS}]", end="\r")

    print(f"  완료: {len(run_results)}개")

    # format / language 성공률
    format_ok = sum(1 for r in run_results if "error" not in r and not [f for f in REQUIRED_FIELDS if f not in r])
    lang_ok = sum(1 for r in run_results
                  if "error" not in r
                  and detect_language(r.get("brand_concept", "") + " " + (r.get("ad_copies") or [{}])[0].get("headline", "")) == EXPECTED_LANG[country])

    # relevance 점수
    valid = [r for r in run_results if "error" not in r]
    relevance_scores = [score_relevance(r) for r in valid]
    rel_avg = sum(relevance_scores) / len(relevance_scores) if relevance_scores else 0
    rel_min = min(relevance_scores) if relevance_scores else 0
    rel_max = max(relevance_scores) if relevance_scores else 0

    # 핵심 메시지 키워드 빈도
    word_counter = Counter()
    for r in valid:
        for msg in r.get("key_messages", []):
            words = [w.strip(".,!?\"'()[]") for w in msg.split() if len(w.strip(".,!?\"'()[]")) >= 2]
            word_counter.update(words)

    all_stats[country] = {
        "n": len(run_results),
        "format_rate": format_ok / len(run_results) * 100,
        "lang_rate": lang_ok / len(run_results) * 100,
        "relevance_avg": rel_avg,
        "relevance_min": rel_min,
        "relevance_max": rel_max,
        "top_keywords": word_counter.most_common(10),
    }

print("\n완료")


[US] 10회 실행 중...
  완료: 10개

[JP] 10회 실행 중...
  완료: 10개

완료


In [7]:
# 통계 출력
for country, stat in all_stats.items():
    print(f"\n{'═' * 50}")
    print(f"  [{country}] {stat['n']}회 실행 결과")
    print(f"{'═' * 50}")

    def bar(pct, total=10):
        filled = round(pct / 100 * total)
        return "█" * filled + "░" * (total - filled)

    def bar5(score, total=10):
        filled = round(score / 5.0 * total)
        return "█" * filled + "░" * (total - filled)

    print(f"  format      {bar(stat['format_rate'])}  {stat['format_rate']:.0f}%")
    print(f"  language    {bar(stat['lang_rate'])}  {stat['lang_rate']:.0f}%")
    print(f"  relevance   {bar5(stat['relevance_avg'])}  {stat['relevance_avg']:.2f}/5.00  (min {stat['relevance_min']:.1f} / max {stat['relevance_max']:.1f})")

    print(f"\n  [핵심 메시지 빈출 키워드]")
    for word, cnt in stat["top_keywords"]:
        print(f"    {word:<15} {'█' * cnt} {cnt}회")


══════════════════════════════════════════════════
  [US] 10회 실행 결과
══════════════════════════════════════════════════
  format      ██████████  100%
  language    ██████████  100%
  relevance   ██████████  4.80/5.00  (min 4.7 / max 4.9)

  [핵심 메시지 빈출 키워드]
    피부              █████████████ 13회
    레티놀과            ████████ 8회
    히알루론산과          ████████ 8회
    개선              ███████ 7회
    콜라겐으로           ███████ 7회
    주름              ██████ 6회
    성분으로            ██████ 6회
    장벽을             ██████ 6회
    나이아신아마이드로       ████ 4회
    장벽              ████ 4회

══════════════════════════════════════════════════
  [JP] 10회 실행 결과
══════════════════════════════════════════════════
  format      ██████████  100%
  language    ██████████  100%
  relevance   ██████████  4.76/5.00  (min 4.6 / max 4.9)

  [핵심 메시지 빈출 키워드]
    피부              ████████████████ 16회
    개선              ████████████ 12회
    강화              ██████████ 10회
    주름              █████████ 9회
    고농축             ███████ 

In [8]:
import json
from openai import OpenAI

client = OpenAI()

RELEVANCE_PROMPT = """아래 제품 정보와 생성된 광고 전략을 보고 relevance를 5.0 만점으로 채점하세요.

relevance: 제품의 성분·효능이 광고 카피와 핵심 메시지에 실제로 반영된 정도

반드시 JSON만 반환하세요:
{"score": 4.2, "reason": "근거 1~2문장"}"""

print(f"{'국가':<6} {'relevance':>10}  이유")
print("─" * 70)

for country, result in results.items():
    if "error" in result:
        continue

    user_msg = f"제품 정보: {json.dumps(SAMPLE, ensure_ascii=False)}\n\n광고 전략: {json.dumps(result, ensure_ascii=False)}"
    resp = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": RELEVANCE_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
    )
    scored = json.loads(resp.choices[0].message.content.strip())
    score = scored.get("score", 0)
    reason = scored.get("reason", "")

    filled = round(score / 5.0 * 10)
    bar = "█" * filled + "░" * (10 - filled)
    print(f"{country:<6} {bar}  {score:.2f}/5.00")
    print(f"       {reason}")

국가      relevance  이유
──────────────────────────────────────────────────────────────────────
US     █████████░  4.70/5.00
       광고 전략이 레티놀·나이아신아마이드·히알루론산·콜라겐·스피큘 등 핵심 성분을 직접 언급하며 미백, 주름개선, 피부결 개선, 보습, 장벽 강화 효능을 전반적으로 잘 반영했습니다. 다만 제품 정보의 '수분 공급'과 '장벽 강화'는 일부 메시지에서만 보이고, 스피큘의 역할 표현이 다소 일반적이라 완전한 일치까지는 아닙니다.
JP     █████████░  4.70/5.00
       광고 카피와 핵심 메시지에 레티놀, 나이아신아마이드, 히알루론산, 스피큘, 콜라겐이 모두 반영되었고 미백·주름개선·피부결 개선·장벽 강화·수분 공급 효능도 전반적으로 잘 연결되었습니다. 다만 일부 문구에서 콜라겐을 탄력 중심으로 확장 해석하거나 레티놀의 표현이 다소 과장되어 보여 제품 정보와 완전히 일치한다고 보긴 어렵습니다.
